# Curated PhysioNet EEGMMIDB 2009 MI Fine-Tuning with SignalJEPA (Within-Subject 5-Fold)

This notebook replaces the old single train/val/test curated EEGMMIDB implementation with the same style as your `physionet2009_finetune_sjepa` notebook: average reference, resample, 0.5-40 Hz filtering, fixed event windows, post-window microvolt scaling, within-subject 5-fold CV, configurable S-JEPA loading, and full artifact saving.


## Configuration notes

- Default path follows your old notebook: `Path.cwd() / "_datasets" / "eegmmidb"`.
- Standard PhysioNet imagined left/right runs are `04`, `08`, `12`. Your old notebook used `02`, `06`, `10`, so this notebook includes a fallback to those runs if no windows are found.
- Default window length is 4.0 seconds, because your curated version exposes 4-second trials.


# 1. Setup


In [ ]:
import os
import random
import sys
from pathlib import Path
import platform
import re
import json
import hashlib
from datetime import datetime
import math

import numpy as np
import pandas as pd

import torch
from torch.utils.data import TensorDataset, Subset

import mne
from mne.channels import make_standard_montage
mne.set_log_level("WARNING")

from braindecode import EEGClassifier
from braindecode.models import SignalJEPA_PreLocal, SignalJEPA_PostLocal, SignalJEPA_Contextual

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, roc_auc_score
from skorch.callbacks import EarlyStopping
from skorch.dataset import ValidSplit

import builtins
print("All imports loaded successfully.")


In [ ]:
print("Runtime Environment:")
print(f"  - Python: {sys.version}")
print(f"  - Platform: {platform.platform()}")

NOTEBOOK_DIR = Path.cwd()
WORKING_DIR = Path.cwd().parent
print(f"\nNotebook dir: {NOTEBOOK_DIR}")
print(f"Working dir:  {WORKING_DIR}")

# Old curated notebook used NOTEBOOK_DIR / "_datasets" / "eegmmidb".
DEFAULT_EEGMMIDB_PATH = NOTEBOOK_DIR / "_datasets" / "eegmmidb"
print(f"Default curated EEGMMIDB path: {DEFAULT_EEGMMIDB_PATH}")


# 2. Configuration


In [ ]:
CONFIG = {
    # Paths: this keeps the old notebook path convention.
    "dataset_path": str(DEFAULT_EEGMMIDB_PATH),
    "artifact_dir": str(NOTEBOOK_DIR / "artifacts" / "curated-eegmmidb-sjepa-5fold"),

    # Subject selection
    "subjects_to_use": None,  # None = all subjects found in curated CSV directory
    "exclude_subjects": [],

    # Curated CSV naming convention
    "file_pattern": r"SUB_(\d+)_(SIG|ANN)_(\d+)\.csv",

    # PhysioNet EEGMMIDB left/right imagined fist runs.
    # Standard PhysioNet imagined left/right runs are 04, 08, 12.
    # Your old notebook used 02, 06, 10; if your curated installation encodes MI that way, set this back.
    "mi_runs": ["04", "08", "12"],
    "fallback_mi_runs_if_empty": ["02", "06", "10"],

    # Raw annotation labels in your old curated implementation.
    # These are mapped to left_hand/right_hand below.
    "raw_label_to_name": {5: "left_hand", 6: "right_hand"},
    "labels_to_keep": ["left_hand", "right_hand"],

    # Signal units in the curated CSV files.
    # If CSV values are already microvolts, keep "microvolts".
    # The code converts to volts for MNE preprocessing, then scales windows back to microvolts.
    "curated_signal_units": "microvolts",  # microvolts or volts

    # Preprocessing/windowing to match your PhysioNet 2009 S-JEPA notebook style.
    "original_sfreq": 160,
    "sfreq": 128,
    "bandpass_low": 0.5,
    "bandpass_high": 40.0,
    "average_reference": True,
    "target_window_duration_s": 4.0,
    "scale_windows_to_microvolts": True,

    # Channel metadata
    "channel_montage": "standard_1020",

    # Model
    "model_name": "SignalJEPA_PreLocal",
    "pretrained_mode": "from_pretrained",  # from_pretrained or random
    "pretrained_repo_id": None,
    "strategy": "new",  # new or full
    "warmup_epochs": 10,

    # Within-subject 5-fold CV
    "cv_folds": 5,
    "val_split": 0.0,

    # Training
    "seed": 12,
    "set_seed": True,
    "batch_size": 32,
    "n_epochs": 50,
    "early_stopping_patience": None,
    "learning_rate": 0.001,
}

LOG_COMPACT = True
PRINT_MODEL_SUMMARY = False
PRINT_FREEZE_DETAILS = False
PRINT_FOLD_CLASS_COUNTS = False


In [ ]:
DATASET_PATH = Path(CONFIG["dataset_path"])
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Curated EEGMMIDB dataset not found at {DATASET_PATH}. "
        "Update CONFIG['dataset_path'] to the directory containing SUB_*_SIG_*.csv and SUB_*_ANN_*.csv."
    )
print(f"Curated EEGMMIDB dataset path: {DATASET_PATH}")
print(f"CSV files available: {len(list(DATASET_PATH.glob('*.csv')))}")

CLASSES_MAPPING = {label: idx for idx, label in enumerate(CONFIG["labels_to_keep"])}
TARGET_N_CLASSES = len(CLASSES_MAPPING)
WINDOW_SAMPLES = int(math.floor(float(CONFIG["target_window_duration_s"]) * float(CONFIG["sfreq"])))
print(f"Class mapping: {CLASSES_MAPPING}")
print(f"Window: {CONFIG['target_window_duration_s']}s -> {WINDOW_SAMPLES} samples at {CONFIG['sfreq']} Hz")


In [ ]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    config_str = json.dumps(CONFIG, sort_keys=True, default=str)
    config_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / CONFIG["model_name"] / CONFIG["strategy"] / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1)

def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop("sep", " ")
    end = kwargs.pop("end", "\n")
    flush = kwargs.pop("flush", False)
    file = kwargs.pop("file", None)
    message = sep.join(str(arg) for arg in args)
    leading_newlines = len(message) - len(message.lstrip("\n"))
    message_body = message[leading_newlines:]
    def _write_target(text):
        if file is None:
            sys.__stdout__.write(text)  # type: ignore
            if flush:
                sys.__stdout__.flush()  # type: ignore
        else:
            file.write(text)
            if flush and hasattr(file, "flush"):
                file.flush()
    if leading_newlines > 0:
        blanks = "\n" * leading_newlines
        _write_target(blanks)
        _LOG_FILE_HANDLE.write(blanks)
    if message_body:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        stamped = f"[{ts}] {message_body}"
        _write_target(stamped + end)
        _LOG_FILE_HANDLE.write(stamped + end)
    else:
        _write_target(end)
        _LOG_FILE_HANDLE.write(end)
    if flush:
        _LOG_FILE_HANDLE.flush()

builtins.print = _timestamped_print

def resolve_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")
DEVICE = resolve_device()
print(f"Using device: {DEVICE}")

def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

BASE_SEED = int(CONFIG["seed"]) if CONFIG["seed"] is not None else None
if CONFIG["set_seed"]:
    seed_everything(BASE_SEED)
    print(f"Seed initialized: {BASE_SEED}")

print("=" * 70)
print("CONFIGURATION")
print("=" * 70)
for key in sorted(CONFIG.keys()):
    print(f"  {key}: {CONFIG[key]}")
print("=" * 70)

config_path = ARTIFACT_DIR / "config.json"
with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)
print(f"Config saved to: {config_path}")


# 3. Load and Prepare Curated EEGMMIDB


In [ ]:
CHANNEL_NAMES = [
    "FC5", "FC3", "FC1", "FCZ", "FC2", "FC4", "FC6",
    "C5", "C3", "C1", "CZ", "C2", "C4", "C6",
    "CP5", "CP3", "CP1", "CPZ", "CP2", "CP4", "CP6",
    "FP1", "FPZ", "FP2",
    "AF7", "AF3", "AFZ", "AF4", "AF8",
    "F7", "F5", "F3", "F1", "FZ", "F2", "F4", "F6", "F8",
    "FT7", "FT8",
    "T7", "T8", "T9", "T10",
    "TP7", "TP8",
    "P7", "P5", "P3", "P1", "PZ", "P2", "P4", "P6", "P8",
    "PO7", "PO3", "POZ", "PO4", "PO8",
    "O1", "OZ", "O2", "IZ",
]

INFO = mne.create_info(
    ch_names=CHANNEL_NAMES,
    sfreq=float(CONFIG["original_sfreq"]),
    ch_types=["eeg"] * len(CHANNEL_NAMES),
)
try:
    montage = make_standard_montage(CONFIG["channel_montage"])
    INFO.set_montage(montage, on_missing="ignore")
except Exception as exc:
    print(f"Montage setup warning: {exc}")

CHS_INFO = INFO["chs"]
print(f"Created MNE Info with {len(CHANNEL_NAMES)} channels at {CONFIG['original_sfreq']} Hz")


In [ ]:
def index_eegmmidb_files(base_path):
    print("Indexing curated EEGMMIDB CSV files...")
    base_path = Path(base_path)
    file_pattern = re.compile(CONFIG["file_pattern"])
    sig_files = {}
    ann_files = {}
    for csv_file in base_path.glob("*.csv"):
        match = file_pattern.search(csv_file.name)
        if not match:
            continue
        subject_id = match.group(1)
        file_type = match.group(2)
        run_id = match.group(3)
        key = (subject_id, run_id)
        if file_type == "SIG":
            sig_files[key] = csv_file
        elif file_type == "ANN":
            ann_files[key] = csv_file
    sig_keys = set(sig_files.keys())
    ann_keys = set(ann_files.keys())
    missing_ann = sig_keys - ann_keys
    missing_sig = ann_keys - sig_keys
    if missing_ann or missing_sig:
        print(f"Warning: missing pairs: missing_ann={sorted(missing_ann)[:10]}, missing_sig={sorted(missing_sig)[:10]}")
    keys = sorted(sig_keys & ann_keys)
    return {key: {"sig_path": sig_files[key], "ann_path": ann_files[key]} for key in keys}

FILE_INDEX = index_eegmmidb_files(DATASET_PATH)
ALL_SUBJECTS_FOUND = sorted({s for s, _ in FILE_INDEX.keys()})
ALL_RUNS_FOUND = sorted({r for _, r in FILE_INDEX.keys()})
print(f"Found paired SIG/ANN files: {len(FILE_INDEX)}")
print(f"Subjects found: {len(ALL_SUBJECTS_FOUND)} | first={ALL_SUBJECTS_FOUND[:5]}")
print(f"Runs found: {ALL_RUNS_FOUND}")

if CONFIG["subjects_to_use"] is None:
    SUBJECTS = ALL_SUBJECTS_FOUND
else:
    SUBJECTS = [f"{int(s):03d}" if str(s).isdigit() else str(s) for s in CONFIG["subjects_to_use"]]
exclude = {f"{int(s):03d}" if str(s).isdigit() else str(s) for s in CONFIG["exclude_subjects"]}
SUBJECTS = [s for s in SUBJECTS if s not in exclude]
print(f"Subjects selected: {len(SUBJECTS)}")

def _load_annotation_rows(ann_path):
    ann = pd.read_csv(ann_path, header=None).values
    return ann

def _raw_label_to_description(raw_label):
    label_map = {int(k): v for k, v in CONFIG["raw_label_to_name"].items()}
    return label_map.get(int(raw_label))

def _signal_to_volts(sig):
    units = str(CONFIG["curated_signal_units"]).lower()
    if units in {"microvolts", "uv", "µv"}:
        return sig.astype(np.float32) * 1e-6
    if units in {"volts", "v"}:
        return sig.astype(np.float32)
    raise ValueError("CONFIG['curated_signal_units'] must be 'microvolts' or 'volts'.")

def build_raw_for_run(sig_path, ann_path):
    sig = pd.read_csv(sig_path, header=None).values
    if sig.shape[1] != len(CHANNEL_NAMES):
        raise ValueError(f"Expected {len(CHANNEL_NAMES)} channels, got {sig.shape[1]} in {sig_path}")
    data_volts = _signal_to_volts(sig).T
    info = INFO.copy()
    raw = mne.io.RawArray(data_volts, info, verbose=False)

    ann_rows = _load_annotation_rows(ann_path)
    onsets = []
    durations = []
    descriptions = []
    for row in ann_rows:
        raw_label = int(row[0])
        desc = _raw_label_to_description(raw_label)
        if desc is None or desc not in CONFIG["labels_to_keep"]:
            continue
        # Old implementation used row[3] as 1-based start and row[4] as end sample at original sfreq.
        start_idx = int(row[3]) - 1
        end_idx = int(row[4])
        if start_idx < 0 or end_idx <= start_idx or end_idx > sig.shape[0]:
            continue
        onsets.append(start_idx / float(CONFIG["original_sfreq"]))
        durations.append((end_idx - start_idx) / float(CONFIG["original_sfreq"]))
        descriptions.append(desc)
    raw.set_annotations(mne.Annotations(onset=onsets, duration=durations, description=descriptions))
    return raw

def preprocess_raw(raw):
    if CONFIG.get("average_reference", True):
        mne.set_eeg_reference(raw, ref_channels="average", copy=False, verbose=False)
    raw.pick_types(eeg=True, meg=False, stim=False)
    raw.resample(float(CONFIG["sfreq"]), verbose=False)
    raw.filter(l_freq=CONFIG["bandpass_low"], h_freq=CONFIG["bandpass_high"], verbose=False)
    return raw

def extract_windows_from_raw(raw):
    X_list, y_list = [], []
    sfreq = float(CONFIG["sfreq"])
    for onset, desc in zip(raw.annotations.onset, raw.annotations.description):
        desc = str(desc)
        if desc not in CLASSES_MAPPING:
            continue
        start = int(round(float(onset) * sfreq))
        stop = start + WINDOW_SAMPLES
        if start < 0 or stop > raw.n_times:
            continue
        x = raw.get_data(start=start, stop=stop).astype(np.float32)
        if CONFIG.get("scale_windows_to_microvolts", True):
            x = x * 1e6
        X_list.append(x)
        y_list.append(CLASSES_MAPPING[desc])
    return X_list, y_list

def extract_subject_trials(subject_id, run_ids):
    X_list, y_list = [], []
    for run_id in run_ids:
        key = (subject_id, run_id)
        if key not in FILE_INDEX:
            continue
        raw = build_raw_for_run(FILE_INDEX[key]["sig_path"], FILE_INDEX[key]["ann_path"])
        raw = preprocess_raw(raw)
        X_run, y_run = extract_windows_from_raw(raw)
        X_list.extend(X_run)
        y_list.extend(y_run)
    if not X_list:
        return np.empty((0, len(CHANNEL_NAMES), WINDOW_SAMPLES), dtype=np.float32), np.empty(0, dtype=np.int64)
    return np.stack(X_list).astype(np.float32), np.asarray(y_list, dtype=np.int64)

def build_subject_data(run_ids):
    subject_data = {}
    for sid in SUBJECTS:
        X, y = extract_subject_trials(sid, run_ids)
        if len(y) > 0:
            subject_data[sid] = (X, y)
            print(f"  Subject {sid}: {len(y)} windows | counts={np.bincount(y, minlength=TARGET_N_CLASSES).tolist()}")
        else:
            print(f"  Subject {sid}: no selected windows")
    return subject_data

print("\nExtracting curated EEGMMIDB MI windows...")
print(f"Primary run IDs: {CONFIG['mi_runs']}")
SUBJECT_DATA = build_subject_data(CONFIG["mi_runs"])

if sum(len(y) for _, y in SUBJECT_DATA.values()) == 0 and CONFIG.get("fallback_mi_runs_if_empty"):
    print("\nNo windows found with primary runs. Trying fallback run IDs...")
    print(f"Fallback run IDs: {CONFIG['fallback_mi_runs_if_empty']}")
    CONFIG["mi_runs_used"] = list(CONFIG["fallback_mi_runs_if_empty"])
    SUBJECT_DATA = build_subject_data(CONFIG["fallback_mi_runs_if_empty"])
else:
    CONFIG["mi_runs_used"] = list(CONFIG["mi_runs"])

if len(SUBJECT_DATA) == 0:
    raise RuntimeError("No subject data extracted. Check mi_runs and raw_label_to_name.")

ALL_X = np.concatenate([X for X, y in SUBJECT_DATA.values()], axis=0)
ALL_y = np.concatenate([y for X, y in SUBJECT_DATA.values()], axis=0)
print("\nDataset summary")
print(f"  Subjects with data: {len(SUBJECT_DATA)}")
print(f"  Total windows:      {len(ALL_y)}")
print(f"  X shape:            {ALL_X.shape}")
print(f"  Class counts:       {np.bincount(ALL_y, minlength=TARGET_N_CLASSES).tolist()}")
print(f"  Runs used:          {CONFIG['mi_runs_used']}")


# 4. Model


In [ ]:
MODEL_REGISTRY = {
    "SignalJEPA_PreLocal": SignalJEPA_PreLocal,
    "SignalJEPA_PostLocal": SignalJEPA_PostLocal,
    "SignalJEPA_Contextual": SignalJEPA_Contextual,
}

DEFAULT_PRETRAINED_REPOS = {
    "SignalJEPA_Contextual": "braindecode/signal-jepa",
    "SignalJEPA_PostLocal": "braindecode/signal-jepa_without-chans",
    "SignalJEPA_PreLocal": "braindecode/signal-jepa_without-chans",
}

def get_model_class(model_name):
    if model_name not in MODEL_REGISTRY:
        raise ValueError(f"Unsupported model_name {model_name}; supported={list(MODEL_REGISTRY)}")
    return MODEL_REGISTRY[model_name]

def get_default_pretrained_repo(model_name):
    return CONFIG["pretrained_repo_id"] or DEFAULT_PRETRAINED_REPOS[model_name]

def build_model(model_name):
    model_cls = get_model_class(model_name)
    n_chans = len(CHANNEL_NAMES)
    n_times = WINDOW_SAMPLES
    mode = CONFIG.get("pretrained_mode", "from_pretrained")
    repo_id = get_default_pretrained_repo(model_name)
    if mode == "from_pretrained":
        model = model_cls.from_pretrained(
            repo_id,
            n_chans=n_chans,
            n_times=n_times,
            n_outputs=TARGET_N_CLASSES,
            strict=False,
        )
        info = {"loading_path": "from_pretrained", "repo_id": repo_id, "mode": mode}
    elif mode == "random":
        model = model_cls(n_chans=n_chans, n_times=n_times, n_outputs=TARGET_N_CLASSES)
        info = {"loading_path": "random_initialization", "repo_id": None, "mode": mode}
    else:
        raise ValueError("CONFIG['pretrained_mode'] must be 'from_pretrained' or 'random'.")
    return model, info

# Verify once that the selected model builds.
test_model, test_info = build_model(CONFIG["model_name"])
print(f"{CONFIG['model_name']} instantiated successfully.")
print(f"  Loading mode:          {test_info['loading_path']}")
print(f"  Repo:                  {test_info.get('repo_id')}")
print(f"  Total parameters:      {sum(p.numel() for p in test_model.parameters()):,}")
print(f"  Trainable parameters:  {sum(p.numel() for p in test_model.parameters() if p.requires_grad):,}")
if PRINT_MODEL_SUMMARY:
    print(test_model)
del test_model

PRETRAINED_CHECKPOINT_INFO = {}

def initialize_model(model_name):
    model, info = build_model(model_name)
    info["model_name"] = model_name
    return model, info


In [ ]:
NEW_LAYER_PREFIXES = {
    "SignalJEPA_PreLocal": ("spatial_conv.", "final_layer."),
    "SignalJEPA_PostLocal": ("final_layer.",),
    "SignalJEPA_Contextual": ("pos_encoder.", "final_layer."),
}

def count_total_and_trainable_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def get_new_layer_prefixes(model_name):
    if model_name not in NEW_LAYER_PREFIXES:
        raise ValueError(f"Unsupported model_name for trainable phase logic: {model_name}")
    return NEW_LAYER_PREFIXES[model_name]

def set_trainable_params_for_phase(model, model_name, phase):
    if phase not in ("new", "warmup", "full"):
        raise ValueError(f"Unsupported phase: {phase}")
    trainable_names = []
    if phase == "full":
        for _, param in model.named_parameters():
            param.requires_grad = True
        trainable_names = [name for name, p in model.named_parameters() if p.requires_grad]
        phase_groups = ["all_parameters"]
    else:
        downstream_prefixes = get_new_layer_prefixes(model_name)
        for _, param in model.named_parameters():
            param.requires_grad = False
        for name, param in model.named_parameters():
            if any(name.startswith(prefix) for prefix in downstream_prefixes):
                param.requires_grad = True
                trainable_names.append(name)
        phase_groups = list(downstream_prefixes)

    total, trainable = count_total_and_trainable_params(model)
    if trainable == 0:
        raise RuntimeError(f"No trainable parameters found for model={model_name}, phase={phase}.")
    return {
        "model_name": model_name,
        "phase": phase,
        "trainable_groups": phase_groups,
        "total_params": int(total),
        "trainable_params": int(trainable),
        "trainable_ratio": float(trainable / total),
        "trainable_names": trainable_names,
    }

def assert_expected_trainable_scope(summary, model_name, phase):
    if phase == "full":
        return
    allowed_prefixes = get_new_layer_prefixes(model_name)
    unexpected_names = [name for name in summary["trainable_names"] if not any(name.startswith(prefix) for prefix in allowed_prefixes)]
    if unexpected_names:
        raise RuntimeError(f"Unexpected trainable parameters for {model_name} phase={phase}: {unexpected_names[:10]}")

def describe_trainable_params(summary, max_names=12):
    print(f"      Trainable groups: {summary['trainable_groups']}")
    print(f"      Total params:     {summary['total_params']:,}")
    print(f"      Trainable params: {summary['trainable_params']:,}")
    print(f"      Trainable pct:    {100.0 * summary['trainable_ratio']:.2f}%")
    if PRINT_FREEZE_DETAILS or not LOG_COMPACT or len(summary["trainable_names"]) <= max_names:
        preview_names = summary["trainable_names"]
    else:
        preview_names = summary["trainable_names"][:max_names]
    print(f"      Trainable parameter names: {preview_names}")
    if len(preview_names) < len(summary["trainable_names"]):
        print(f"      ... {len(summary['trainable_names']) - len(preview_names)} additional trainable parameters hidden")


# 5. Training


In [ ]:
def get_targets(dataset):
    return np.asarray([int(dataset[i][1]) for i in range(len(dataset))], dtype=np.int64)

def make_train_split():
    val_split = CONFIG.get("val_split", 0.0)
    if val_split is None or float(val_split) <= 0.0:
        return None
    return ValidSplit(cv=float(val_split), stratified=True, random_state=12)

def make_callbacks():
    train_split = make_train_split()
    patience = CONFIG.get("early_stopping_patience", None)
    if train_split is None or patience is None or int(patience) <= 0:
        return []
    return [
        (
            "early_stopping",
            EarlyStopping(
                monitor="valid_loss",
                patience=int(patience),
                lower_is_better=True,
                load_best=True,
            ),
        )
    ]

def build_classifier(model, callbacks, max_epochs, fold_seed=None, warm_start=False):
    train_generator = None
    if fold_seed is not None:
        train_generator = torch.Generator()
        train_generator.manual_seed(fold_seed)
    clf_kwargs = {
        "batch_size": CONFIG["batch_size"],
        "max_epochs": int(max_epochs),
        "device": DEVICE,
        "callbacks": callbacks,
        "train_split": make_train_split(),
        "classes": range(TARGET_N_CLASSES),
        "iterator_train__shuffle": True,
        "iterator_train__num_workers": 0,
        "iterator_valid__num_workers": 0,
        "optimizer": torch.optim.Adam,
        "warm_start": warm_start,
    }
    if CONFIG["learning_rate"] is not None:
        clf_kwargs["lr"] = CONFIG["learning_rate"]
    if train_generator is not None:
        clf_kwargs["iterator_train__generator"] = train_generator
    return EEGClassifier(model, **clf_kwargs)

def run_one_batch_finite_sanity_check(model, train_set, model_name):
    if len(train_set) == 0:
        raise RuntimeError(f"{model_name}: train_set is empty during sanity check.")
    batch_size = int(min(CONFIG["batch_size"], len(train_set)))
    sanity_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=False, num_workers=0)
    batch = next(iter(sanity_loader))
    x_batch = torch.as_tensor(batch[0]).float().to(DEVICE)
    y_batch = torch.as_tensor(batch[1]).long().to(DEVICE)
    was_training = model.training
    model = model.to(DEVICE)
    model.eval()
    with torch.no_grad():
        logits = model(x_batch)
        if not torch.isfinite(logits).all():
            raise RuntimeError(f"{model_name}: non-finite logits detected.")
        loss = torch.nn.functional.cross_entropy(logits, y_batch)
        if not torch.isfinite(loss):
            raise RuntimeError(f"{model_name}: non-finite loss detected.")
    if was_training:
        model.train()
    print(f"    Sanity check passed: finite logits/loss on one training batch for {model_name}.")

def extract_binary_score_vector(score_output, expected_n_samples):
    if score_output is None:
        return None
    scores = np.asarray(score_output)
    if scores.ndim == 1:
        score_vec = scores.astype(float)
    elif scores.ndim == 2 and scores.shape[1] == 2:
        score_vec = scores[:, 1].astype(float)
    elif scores.ndim == 2 and scores.shape[1] == 1:
        score_vec = scores[:, 0].astype(float)
    else:
        return None
    if score_vec.shape[0] != int(expected_n_samples) or not np.isfinite(score_vec).all():
        return None
    return score_vec

def compute_classification_metrics(y_true, y_pred, y_score=None, paradigm=None):
    y_true = np.asarray(y_true).astype(int).reshape(-1)
    y_pred = np.asarray(y_pred).astype(int).reshape(-1)
    metrics = {"accuracy": None, "balanced_accuracy": None, "roc_auc": None}
    metrics["accuracy"] = float(accuracy_score(y_true, y_pred))
    metrics["balanced_accuracy"] = float(balanced_accuracy_score(y_true, y_pred))
    if paradigm == "ERP" and y_score is not None and len(np.unique(y_true)) >= 2:
        metrics["roc_auc"] = float(roc_auc_score(y_true, y_score))
    return metrics

def run_training_and_eval(train_set, test_set, fold_id, fold_label, n_total_folds=None):
    global PRETRAINED_CHECKPOINT_INFO
    if CONFIG["set_seed"]:
        fold_seed = BASE_SEED
        seed_everything(fold_seed)  # type: ignore
    else:
        fold_seed = None

    y_train = get_targets(train_set)
    y_test = get_targets(test_set)
    train_counts = np.bincount(y_train, minlength=TARGET_N_CLASSES)
    test_counts = np.bincount(y_test, minlength=TARGET_N_CLASSES)

    model_name = CONFIG["model_name"]
    strategy = CONFIG["strategy"]
    warmup_epochs = int(CONFIG["warmup_epochs"])
    model, pretrained_load_summary = initialize_model(model_name)
    PRETRAINED_CHECKPOINT_INFO = dict(pretrained_load_summary)

    total_folds_text = f"/{n_total_folds}" if n_total_folds is not None else ""
    print(f"\nFold {fold_id}{total_folds_text} | {fold_label}")
    print(f"    Train windows:           {len(train_set)} | counts={train_counts.tolist()}")
    print(f"    Test windows:            {len(test_set)} | counts={test_counts.tolist()}")
    print(f"    Downstream model:        {model_name}")
    print(f"    Fine-tune strategy:      {strategy}")
    print(f"    Pretrained loading path: {pretrained_load_summary['loading_path']}")
    print(f"    Pretrained repo:         {pretrained_load_summary.get('repo_id')}")

    if strategy == "new":
        phase_1_summary = set_trainable_params_for_phase(model, model_name, "new")
        assert_expected_trainable_scope(phase_1_summary, model_name, "new")
        print("    Phase 1 (new):")
        describe_trainable_params(phase_1_summary)
        run_one_batch_finite_sanity_check(model, train_set, model_name)
        clf = build_classifier(model, callbacks=make_callbacks(), max_epochs=int(CONFIG["n_epochs"]), fold_seed=fold_seed, warm_start=False)
        phase_summaries = {"phase_1": phase_1_summary, "phase_2": None}
        clf.fit(train_set, y=y_train)
    elif strategy == "full":
        if warmup_epochs < 1:
            raise ValueError("For strategy='full', warmup_epochs must be >= 1.")
        phase_1_summary = set_trainable_params_for_phase(model, model_name, "warmup")
        assert_expected_trainable_scope(phase_1_summary, model_name, "warmup")
        print("    Phase 1 (warmup):")
        describe_trainable_params(phase_1_summary)
        run_one_batch_finite_sanity_check(model, train_set, model_name)
        clf = build_classifier(model, callbacks=[], max_epochs=warmup_epochs, fold_seed=fold_seed, warm_start=True)
        clf.fit(train_set, y=y_train)
        phase_2_summary = set_trainable_params_for_phase(clf.module_, model_name, "full")
        print("    Phase 2 (full):")
        describe_trainable_params(phase_2_summary)
        clf.initialize_optimizer()
        remaining_epochs = int(CONFIG["n_epochs"]) - warmup_epochs
        if remaining_epochs < 1:
            raise ValueError("CONFIG['n_epochs'] must be greater than warmup_epochs for strategy='full'.")
        clf.set_params(callbacks=make_callbacks(), max_epochs=remaining_epochs)
        clf.fit(train_set, y=y_train)
        phase_summaries = {"phase_1": phase_1_summary, "phase_2": phase_2_summary}
    else:
        raise ValueError("CONFIG['strategy'] must be 'new' or 'full'.")

    y_pred = clf.predict(test_set)
    y_score_raw = None
    if hasattr(clf, "predict_proba"):
        try:
            y_score_raw = clf.predict_proba(test_set)
        except Exception:
            y_score_raw = None
    y_score = extract_binary_score_vector(y_score_raw, expected_n_samples=len(y_test)) if CONFIG["paradigm"] == "ERP" else None
    metrics = compute_classification_metrics(y_test, y_pred, y_score=y_score, paradigm=CONFIG["paradigm"])

    stopped_epoch = int(clf.history[-1]["epoch"]) if len(clf.history) > 0 else 0  # type: ignore
    valid_loss_curve = [(int(row["epoch"]), float(row["valid_loss"])) for row in clf.history if "valid_loss" in row]
    if valid_loss_curve:
        best_epoch, best_valid_loss = min(valid_loss_curve, key=lambda x: x[1])
    else:
        best_epoch, best_valid_loss = None, None

    cm = confusion_matrix(y_test, y_pred, labels=np.arange(TARGET_N_CLASSES)).tolist()
    pred_hist = np.bincount(y_pred, minlength=TARGET_N_CLASSES).tolist()
    acc = metrics["accuracy"]
    bal_acc = metrics["balanced_accuracy"]
    roc_auc = metrics["roc_auc"]
    print(f"    Result | best_epoch={best_epoch} | stop={stopped_epoch} | acc={acc:.4f} | bal_acc={bal_acc:.4f} | pred_hist={pred_hist}")

    return {
        "fold_id": int(fold_id),
        "fold_label": str(fold_label),
        "paradigm": CONFIG["paradigm"],
        "model_name": model_name,
        "strategy": strategy,
        "warmup_epochs": warmup_epochs,
        "n_train": len(train_set),
        "n_test": len(test_set),
        "train_class_counts": train_counts.tolist(),
        "test_class_counts": test_counts.tolist(),
        "pretrained_load": pretrained_load_summary,
        "phase_1_trainable_groups": phase_summaries["phase_1"]["trainable_groups"],
        "phase_1_trainable_params": phase_summaries["phase_1"]["trainable_params"],
        "phase_1_trainable_names": phase_summaries["phase_1"]["trainable_names"],
        "phase_2_trainable_groups": None if phase_summaries["phase_2"] is None else phase_summaries["phase_2"]["trainable_groups"],
        "phase_2_trainable_params": None if phase_summaries["phase_2"] is None else phase_summaries["phase_2"]["trainable_params"],
        "phase_2_trainable_names": None if phase_summaries["phase_2"] is None else phase_summaries["phase_2"]["trainable_names"],
        "best_epoch": best_epoch,
        "stopped_epoch": stopped_epoch,
        "best_valid_loss": best_valid_loss,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "roc_auc": roc_auc,
        "confusion_matrix": cm,
        "prediction_histogram": pred_hist,
    }


In [ ]:
def make_subject_dataset(X, y):
    return TensorDataset(torch.as_tensor(X, dtype=torch.float32), torch.as_tensor(y, dtype=torch.long))

def make_fold_splits(y, n_folds, n_classes):
    indices = np.arange(len(y))
    counts = np.bincount(y, minlength=n_classes)
    if counts.min() < n_folds:
        raise ValueError(f"Cannot use {n_folds} folds with class counts={counts.tolist()}.")
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=12)
    return [
        {"fold_id": fold_id, "idx_train": train_idx, "idx_test": test_idx}
        for fold_id, (train_idx, test_idx) in enumerate(skf.split(indices, y), start=1)
    ]

def run_subject_cv(subject_id, X, y):
    subject_dataset = make_subject_dataset(X, y)
    counts = np.bincount(y, minlength=TARGET_N_CLASSES)
    print(f"\nSubject {subject_id}: {len(y)} windows | class_counts={counts.tolist()}")
    folds = make_fold_splits(y, CONFIG["cv_folds"], TARGET_N_CLASSES)
    results = []
    for fold in folds:
        train_set = Subset(subject_dataset, fold["idx_train"].tolist())
        test_set = Subset(subject_dataset, fold["idx_test"].tolist())
        result = run_training_and_eval(train_set, test_set, fold["fold_id"], f"subject={subject_id}", n_total_folds=CONFIG["cv_folds"])
        result["subject_id"] = str(subject_id)
        results.append(result)
    accs = [r["accuracy"] for r in results if r["accuracy"] is not None]
    bals = [r["balanced_accuracy"] for r in results if r["balanced_accuracy"] is not None]
    print(f"  Subject {subject_id} summary: acc={np.mean(accs):.4f}±{np.std(accs):.4f}  bal_acc={np.mean(bals):.4f}±{np.std(bals):.4f}")
    return results

print("=" * 70)
print("STARTING CURATED EEGMMIDB WITHIN-SUBJECT 5-FOLD CV")
print("=" * 70)
print(f"Subjects:      {list(SUBJECT_DATA.keys())}")
print(f"Runs used:     {CONFIG['mi_runs_used']}")
print(f"Model:         {CONFIG['model_name']}")
print(f"Pretrained:    {CONFIG['pretrained_mode']}")
print(f"Strategy:      {CONFIG['strategy']}")
print(f"CV folds:      {CONFIG['cv_folds']}")
print(f"Window:        {CONFIG['target_window_duration_s']}s / {WINDOW_SAMPLES} samples")
print("=" * 70)

FOLD_RESULTS = []
for sid, (X, y) in SUBJECT_DATA.items():
    FOLD_RESULTS.extend(run_subject_cv(sid, X, y))
print(f"\nTotal folds completed: {len(FOLD_RESULTS)}")


# 6. Results


In [ ]:
def aggregate_results(fold_results):
    # Per-subject aggregation when subject_id exists; per-heldout aggregation for LOSO.
    group_key = "subject_id" if any("subject_id" in r for r in fold_results) else "held_out_subject_id"
    grouped = {}
    for result in fold_results:
        gid = result.get(group_key, "global")
        grouped.setdefault(gid, {"accuracies": [], "balanced_accuracies": [], "roc_aucs": []})
        grouped[gid]["accuracies"].append(result.get("accuracy"))
        grouped[gid]["balanced_accuracies"].append(result.get("balanced_accuracy"))
        grouped[gid]["roc_aucs"].append(result.get("roc_auc"))
    for gid, m in grouped.items():
        acc_values = [v for v in m["accuracies"] if v is not None]
        bal_values = [v for v in m["balanced_accuracies"] if v is not None]
        roc_values = [v for v in m["roc_aucs"] if v is not None]
        m["mean_accuracy"] = float(np.mean(acc_values)) if acc_values else None
        m["std_accuracy"] = float(np.std(acc_values)) if acc_values else None
        m["mean_balanced_accuracy"] = float(np.mean(bal_values)) if bal_values else None
        m["std_balanced_accuracy"] = float(np.std(bal_values)) if bal_values else None
        m["mean_roc_auc"] = float(np.mean(roc_values)) if roc_values else None
        m["std_roc_auc"] = float(np.std(roc_values)) if roc_values else None

    all_accs = [r["accuracy"] for r in fold_results if r.get("accuracy") is not None]
    all_bals = [r["balanced_accuracy"] for r in fold_results if r.get("balanced_accuracy") is not None]
    all_rocs = [r["roc_auc"] for r in fold_results if r.get("roc_auc") is not None]
    global_metrics = {
        "mean_accuracy": float(np.mean(all_accs)) if all_accs else None,
        "std_accuracy": float(np.std(all_accs)) if all_accs else None,
        "mean_balanced_accuracy": float(np.mean(all_bals)) if all_bals else None,
        "std_balanced_accuracy": float(np.std(all_bals)) if all_bals else None,
        "mean_roc_auc": float(np.mean(all_rocs)) if all_rocs else None,
        "std_roc_auc": float(np.std(all_rocs)) if all_rocs else None,
        "n_groups": len(grouped),
        "n_folds_total": len(fold_results),
    }
    return grouped, global_metrics

SUBJECT_METRICS, GLOBAL_METRICS = aggregate_results(FOLD_RESULTS)

print("=" * 70)
print("AGGREGATED RESULTS")
print("=" * 70)
for gid, m in sorted(SUBJECT_METRICS.items(), key=lambda x: _sort_subject_key(x[0])):
    acc_str = f"{m['mean_accuracy']:.4f}±{m['std_accuracy']:.4f}" if m["mean_accuracy"] is not None else "N/A"
    bal_str = f"{m['mean_balanced_accuracy']:.4f}±{m['std_balanced_accuracy']:.4f}" if m["mean_balanced_accuracy"] is not None else "N/A"
    print(f"  {gid}: acc={acc_str}  bal_acc={bal_str}")
print("-" * 70)
print(f"  OVERALL: acc={GLOBAL_METRICS['mean_accuracy']:.4f}±{GLOBAL_METRICS['std_accuracy']:.4f}  bal_acc={GLOBAL_METRICS['mean_balanced_accuracy']:.4f}±{GLOBAL_METRICS['std_balanced_accuracy']:.4f}")
print("=" * 70)

cv_results_path = ARTIFACT_DIR / "cv_results.json"
with open(cv_results_path, "w") as f:
    json.dump(FOLD_RESULTS, f, indent=2)
subject_metrics_path = ARTIFACT_DIR / "subject_metrics.json"
with open(subject_metrics_path, "w") as f:
    json.dump(SUBJECT_METRICS, f, indent=2)
global_metrics_path = ARTIFACT_DIR / "global_metrics.json"
with open(global_metrics_path, "w") as f:
    json.dump(GLOBAL_METRICS, f, indent=2)

run_metadata = {
    "run_id": RUN_ID,
    "artifact_dir": str(ARTIFACT_DIR),
    "dataset_name": "Curated_EEGMMIDB",
    "dataset_source": "curated_csv",
    "dataset_path": str(DATASET_PATH),
    "mi_runs_used": list(CONFIG.get("mi_runs_used", [])),
    "labels_to_keep": list(CONFIG["labels_to_keep"]),
    "excluded_subjects": list(CONFIG["exclude_subjects"]),
    "target_window_duration_s": TARGET_TRIAL_DURATION_S,
    "window_samples": WINDOW_SAMPLES,
    "subjects": [str(s) for s in SUBJECT_DATA.keys()],
    "model_name": CONFIG["model_name"],
    "pretrained_mode": CONFIG["pretrained_mode"],
    "strategy": CONFIG["strategy"],
    "warmup_epochs": int(CONFIG["warmup_epochs"]),
    "pretrained_checkpoint_info": PRETRAINED_CHECKPOINT_INFO,
    "cv_seed": BASE_SEED,
    "global_metrics": GLOBAL_METRICS,
}
run_metadata_path = ARTIFACT_DIR / "run_metadata.json"
with open(run_metadata_path, "w") as f:
    json.dump(run_metadata, f, indent=2)

print(f"CV results saved to:      {cv_results_path}")
print(f"Subject metrics saved to: {subject_metrics_path}")
print(f"Global metrics saved to:  {global_metrics_path}")
print(f"Run metadata saved to:    {run_metadata_path}")
print(f"\nAll artifacts in: {ARTIFACT_DIR}")

print("\n" + "=" * 70)
print("EXPERIMENT SUMMARY")
print("=" * 70)
print(f"Run ID:                 {RUN_ID}")
print("Dataset:                Curated_EEGMMIDB")
print(f"Labels kept:            {CONFIG['labels_to_keep']}")
print(f"Model:                  {CONFIG['model_name']}")
print(f"Pretrained mode:        {CONFIG['pretrained_mode']}")
print(f"Strategy:               {CONFIG['strategy']}")
print(f"Window:                 {TARGET_TRIAL_DURATION_S:.3f}s / {WINDOW_SAMPLES} samples")
print(f"Artifacts:              {ARTIFACT_DIR}")
print(f"Mean Accuracy:          {GLOBAL_METRICS['mean_accuracy']:.4f} ± {GLOBAL_METRICS['std_accuracy']:.4f}")
print(f"Mean Balanced Accuracy: {GLOBAL_METRICS['mean_balanced_accuracy']:.4f} ± {GLOBAL_METRICS['std_balanced_accuracy']:.4f}")
print("=" * 70)
